# SurgeFlow live monitor

The monitor runs every available offering on its cadence. Offerings 03 and 09 remain unavailable by member key.

The default Hotlist market is `us`. Fixed weekday exchange hours gate that panel; holidays are not handled.

Each panel refreshes on a schedule; each panel shows its source date. Changed rows receive a highlight.

Research data only, not investment advice. Free access grants no redistribution rights. **Do not republish the data you fetch.**

Fixture mode runs one full refresh and stops. Run the Stop cell to end a live monitor.

In [ ]:
import asyncio
import sys
from datetime import datetime, timezone
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetched_at, fixture_mode, get_api_key, make_client
from surgeflow.monitor import Monitor, RequestBudget, default_panels
client, catalogue = make_client()
api_key = get_api_key()
FIXTURE_NOW = datetime(2026, 11, 23, 15, 0, tzinfo=timezone.utc)
def monitor_now():
    return FIXTURE_NOW if fixture_mode() else datetime.now(timezone.utc)
budget = RequestBudget(daily_limit=1500)
budget.consume(1, monitor_now())
try:
    me = client.member_me(api_key)
    print('Plan:', me.get('plan'), '; usage:', me.get('usage'))
except Exception as exc:
    print('Account check error:', type(exc).__name__, str(exc), '; fetched at:', fetched_at())
monitor = Monitor(default_panels(client, api_key), budget=budget, now=monitor_now)
projected = monitor.print_projection()
print('Projected total including one startup account check:', projected + 1)
print('Technical cohorts: unavailable by member key; no panel request.')
print('Layoffs: unavailable by member key; no panel request.')

In [ ]:
STOP = False
async def refresh_loop():
    while not STOP:
        monitor.run_due()
        monitor.render()
        if fixture_mode():
            break
        await asyncio.sleep(1)
monitor_task = asyncio.create_task(refresh_loop())
await asyncio.sleep(0)

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print('Optional ipywidgets is not installed. Use the Stop cell.')
else:
    stop_button = widgets.Button(description='Stop monitor')
    def stop_button_clicked(_):
        global STOP
        STOP = True
    stop_button.on_click(stop_button_clicked)
    display(stop_button)

## Stop
Run the next cell to stop the refresh loop.

In [ ]:
STOP = True
if not monitor_task.done():
    monitor_task.cancel()
print('Stop requested. Requests used locally:', monitor.budget.used, '/', monitor.budget.daily_limit)